# Simulating the BB84 Quantum Key Distribution Protocol

Welcome to this notebook, where we'll explore the fascinating BB84 Quantum Key Distribution (QKD) protocol. This protocol allows two parties, traditionally named Alice and Bob, to establish a secret cryptographic key with provable security, even in the presence of an eavesdropper, Eve.

Our goal here is not just to see BB84 work, but specifically to demonstrate how an *eavesdropper* (Eve) attempting to intercept the quantum communication will inevitably introduce detectable errors. This detection mechanism is a cornerstone of quantum cryptography.

Let's set up our environment and dive into the quantum world!

In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 103.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=c17964cf7d2b1561d82642a99edfdd93c1395bf7e41fd65a58fb3bc167722a0a
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator

NUM_QUBITS = 24 # Reduced to 24, the maximum for BasicSimulator

def qrng(n):
    # generate random bits using quantum superposition
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    qc.measure(range(n), range(n))

    sim = BasicSimulator()
    result = sim.run(transpile(qc, sim), shots=1).result()

    # reverse bitstring so it matches standard 0 to n-1 indexing
    bits = list(result.get_counts().keys())[0][::-1]
    return [int(b) for b in bits]

# --- Alice prepares her qubits ---
print("Alice is preparing the qubits...")
alice_bits = qrng(NUM_QUBITS)
alice_bases = qrng(NUM_QUBITS) # 0 is Z-basis, 1 is X-basis

encoded_qubits = []
for i in range(NUM_QUBITS):
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1:
        qc.x(0)
    if alice_bases[i] == 1:
        qc.h(0)
    encoded_qubits.append(qc)

# --- Eve intercepts the transmission ---
print("Eve intercepts the qubits and measures them...")
eve_bases = qrng(NUM_QUBITS)
intercepted_qubits = []

sim = BasicSimulator()
for i in range(NUM_QUBITS):
    qc = encoded_qubits[i]

    # Eve measures in her own random basis
    if eve_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)

    res = sim.run(transpile(qc, sim), shots=1).result()
    measured_val = int(list(res.get_counts().keys())[0])

    # Eve creates a new qubit to try and fool Bob
    new_qc = QuantumCircuit(1, 1)
    if measured_val == 1:
        new_qc.x(0)
    if eve_bases[i] == 1:
        new_qc.h(0)

    intercepted_qubits.append(new_qc)

# --- Bob measures the qubits (unknowingly receiving Eve's fakes) ---
print("Bob is measuring the received qubits...")
bob_bases = qrng(NUM_QUBITS)
bob_results = []

for i in range(NUM_QUBITS):
    qc = intercepted_qubits[i]

    # Apply H gate if Bob decides to measure in the X-basis
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure(0, 0)

    res = sim.run(transpile(qc, sim), shots=1).result()
    measured_val = int(list(res.get_counts().keys())[0])
    bob_results.append(measured_val)

# --- Sifting the keys ---
print("Comparing bases over public channel...")
alice_key = []
bob_key = []

for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_results[i])

print(f"Sifted key length: {len(alice_key)}")

# --- Security check ---
# For demonstration purposes, we'll check the error rate across the entire sifted key.
# In a real protocol, only a random subset would be checked publicly.
sample_size = len(alice_key) # Use the entire sifted key for error calculation
errors = 0

for i in range(sample_size):
    if alice_key[i] != bob_key[i]:
        errors += 1

error_rate = errors / sample_size if sample_size > 0 else 0
print(f"Error rate on tested subset: {error_rate * 100:.2f}%") # Format to 2 decimal places

# The theoretical error rate with an intercept-resend attack is 25%.
# We use a 10% threshold to account for statistical variance.
THRESHOLD = 0.10 # 10% threshold to allow for slight simulation variations

if error_rate <= THRESHOLD:
    print("No eavesdropper detected. Key is secure.")
else:
    print("Eavesdropper detected! High error rate found. Protocol aborted.")

Alice is preparing the qubits...
Eve intercepts the qubits and measures them...
Bob is measuring the received qubits...
Comparing bases over public channel...
Sifted key length: 17
Error rate on tested subset: 29.41%
Eavesdropper detected! High error rate found. Protocol aborted.
